# Market Information Diffusion — guided walkthrough

This notebook explains the model, runs an explicitly synthetic example, and inspects the aggregate real-study findings. Synthetic observations validate software behavior; they do not establish empirical conclusions.


In [1]:
from pathlib import Path
import sys, json, math, tempfile
# Work from either the repository root or its notebooks directory.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src/market_diffusion').is_dir())
sys.path.insert(0, str(ROOT / 'src'))


## 1. Define event time and surprise

Event time is seconds since the release. Surprise is actual minus pre-release consensus. Standardization uses only previous same-series surprises; future releases must never enter the scale estimate.


In [2]:
actual, consensus = 0.4, 0.3
surprise = actual - consensus  # Percentage points, not a percentage return.
print(f"Illustrative surprise: {surprise:.2f} percentage points")


Illustrative surprise: 0.10 percentage points


## 2. Understand the curve

Normalize log-midpoint movement by a retrospective endpoint move. The fitted curve is D(t)=1−(1−j)exp(−t/k). j is the initial fraction; k is a time scale. The residual half-life differs from t50 when j is nonzero. These illustrative parameters are not fitted empirical estimates.


In [3]:
j, k = 0.4, 40.0
response = lambda t: 1 - (1-j) * math.exp(-t/k)
t50 = 0.0 if j >= 0.5 else k * math.log((1-j)/0.5)
print({"illustrative_t50_seconds": t50, "residual_half_life_seconds": k*math.log(2)})
print([(t, round(response(t), 3)) for t in [0, 1, 5, 30, 60, 300]])


{'illustrative_t50_seconds': 7.2928622717581835, 'residual_half_life_seconds': 27.725887222397812}
[(0, 0.4), (1, 0.415), (5, 0.471), (30, 0.717), (60, 0.866), (300, 1.0)]


## 3. Run the actual pipeline on labelled synthetic data

The same core analysis handles the demo and supplied observations. A fresh temporary directory prevents overwriting earlier runs. Coverage, small-move and fit checks remain active.


In [4]:
from market_diffusion.demo import generate
from market_diffusion.pipeline import run
from market_diffusion.alignment import Config
workspace = Path(tempfile.mkdtemp(prefix='diffusion-notebook-'))
generate(workspace / 'inputs', seed=42)
simulated = run(workspace/'inputs/events.csv', workspace/'inputs/quotes.csv', workspace/'analysis', Config(), 'simulation')
print("SIMULATION status counts:", simulated['counts'])


SIMULATION status counts: {'accepted': 46, 'review': 2}


## 4. Inspect the real-study aggregate evidence

The real study has 35 dates × two currencies. An accepted response passes implemented screens; it is not proof of causality. Separate currency medians use different accepted samples. Only one same-date pair survives.


In [5]:
empirical = json.loads((ROOT / 'docs/study-summary.json').read_text())
for phase, result in empirical.items():
    print(phase, result['counts'])
print("Paired timing evidence:", empirical['primary']['paired_comparisons'])
assert sum(empirical['primary']['counts'].values()) == 70


primary {'excluded': 48, 'review': 17, 'accepted': 5}
development {'excluded': 36, 'review': 14, 'accepted': 4}
holdout {'review': 3, 'accepted': 1, 'excluded': 12}
Paired timing evidence: {'EURUSD − GBPUSD': {'n': 1, 'median': -6.468914209123743, 'ci_low': None, 'ci_high': None}}


## 5. Interpret and extend responsibly

![Primary screening](../docs/figures/screening.png)

The result is model/data fragility, not a trading edge. Forty-four windows fail coverage; four have unstable normalization. The 2025 sample contains one accepted response. Read the [report](../docs/research-report.md) and [reproduction instructions](../docs/reproduction.md) before changing thresholds. Future research needs better quote coverage and independent evaluation; relaxing a screen after seeing results changes the research question.
